In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction==2026.11.14 --extra-index-url https://download.easydiffraction.org/pypi/simple/


# Co₂SiO₄ on D20, cooling scan

&nbsp; &nbsp; &nbsp; &nbsp; &nbsp; &nbsp; &nbsp; 

Fit the olivine Co₂SiO₄ with its magnetic structure to the whole D20 cooling scan, 213 patterns
from 497.4 K down to 2.3 K, one after another and from the lowest temperature up. First the fit
is staged on the 2.3 K pattern, the scan's template; then each later pattern starts from the
previous result. The cobalt moments stay free at every temperature and fall towards zero near
the Néel temperature, about 50 K.

It builds the catalogue case `pd-neut-cwl_cosio-d20_scan-213f` step by step: the same project the
command line fits and the app's Examples open. Its two Co sites share one displacement.

## Import the library

In [2]:
import csv
import zipfile
from pathlib import Path

import easydiffraction as edi

## Define the project

A new project is empty: it holds no structure and no experiment until you add them.

In [3]:
project = edi.Project(name='pd-neut-cwl_cosio-d20_scan-213f')

## Define the structure

Load the structure from EasyDiffraction's collection. Its CIF gives the space group, the cell and
the atom sites.

In [4]:
path = edi.download_data('pd-neut-cwl_cosio-d20_scan-213f-cosio', destination='data')
project.structures.add_from_cif_path(path)
structure = project.structures['cosio']

Data 'pd-neut-cwl_cosio-d20_scan-213f-cosio' is at data/pd-neut-cwl_cosio-d20_scan-213f-cosio.edi


In [5]:
project.display.structure('cosio')

View(html=<323,203 characters>, height=660)

## Define the experiment

Create the experiment from the measured data in EasyDiffraction's collection, then describe the
instrument, the peak profile, the background and the structure it links.

In [6]:
data_path = edi.download_data('pd-neut-cwl_cosio-d20_scan-213f-d20-data', destination='data')
project.experiments.add_from_data_path(
    name='d20',
    data_path=data_path,
    sample_form='powder',
    beam_mode='constant wavelength',
    radiation_probe='neutron',
    scattering_type='bragg',
)
experiment = project.experiments['d20']
experiment.peak.type = 'cwl-tch-pseudo-voigt-fcj'
experiment.absorption.type = 'cylinder-hewat'
experiment.instrument.name = 'D20'
experiment.instrument.facility = 'ILL'
experiment.peak.broad_gauss_u = 0.273527
experiment.peak.broad_gauss_v = -0.589718
experiment.peak.broad_gauss_w = 0.413007
experiment.peak.broad_lorentz_y = 0.0114
experiment.peak.asym_fcj_1 = 0.0619
experiment.peak.asym_fcj_2 = 0.0619
experiment.peak.cutoff_fwhm = 8
experiment.absorption.mu_r = 0.230
experiment.instrument.setup_wavelength = 1.87
experiment.instrument.calib_twotheta_offset = 0.29433
experiment.excluded_regions.create(start=0, end=6)
experiment.excluded_regions.create(start=64.65, end=64.74)
experiment.excluded_regions.create(start=150, end=180)
experiment.linked_structures.create(structure_id='cosio', scale=1.817226)
experiment.background.create(position=6, intensity=603.5)
experiment.background.create(position=6.5, intensity=585.0)
experiment.background.create(position=7, intensity=568.5)
experiment.background.create(position=7.6, intensity=528.8)
experiment.background.create(position=12.6, intensity=418.9)
experiment.background.create(position=13, intensity=420.2)
experiment.background.create(position=13.5, intensity=416.5)
experiment.background.create(position=15, intensity=416.0542)
experiment.background.create(position=25, intensity=394.7747)
experiment.background.create(position=30, intensity=373.1628)
experiment.background.create(position=50, intensity=406.3737)
experiment.background.create(position=70, intensity=408.5056)
experiment.background.create(position=90, intensity=395.9971)
experiment.background.create(position=110, intensity=345.9223)
experiment.background.create(position=130, intensity=279.0409)
experiment.background.create(position=150, intensity=230.6987)

Data 'pd-neut-cwl_cosio-d20_scan-213f-d20-data' is at data/pd-neut-cwl_cosio-d20_scan-213f-d20.xye


The constraints the fit keeps, each between two named parameters (aliases).

In [7]:
project.analysis.aliases.create(id='biso_Co1', param=structure.atom_sites['Co1'].adp_iso)
project.analysis.aliases.create(id='biso_Co2', param=structure.atom_sites['Co2'].adp_iso)
project.analysis.constraints.create(id='biso_Co2', expression='biso_Co2 = biso_Co1')

In [8]:
project.save_as('pd-neut-cwl_cosio-d20_scan-213f')

Saving project to 'pd-neut-cwl_cosio-d20_scan-213f'


The scan's patterns come from the collection as one archive. Unpack them into the project's
folder and declare them as the scan the sequential fit runs over, from the last file up; it reads
each pattern's temperature from its header.

In [9]:
scan_path = edi.download_data('pd-neut-cwl_cosio-d20_scan-213f-scan', destination='data')
with zipfile.ZipFile(scan_path) as archive:
    archive.extractall('pd-neut-cwl_cosio-d20_scan-213f/experiments/d20_scan')
project.analysis.sequential_fit.data_dir = 'experiments/d20_scan'
project.analysis.sequential_fit.file_pattern = '*.dat'
project.analysis.sequential_fit.reverse = True
project.analysis.sequential_fit_extract.create(
    id='temperature', target='diffrn.ambient_temperature', pattern=r'^TEMP\s+([0-9.]+)'
)
project.save()

Data 'pd-neut-cwl_cosio-d20_scan-213f-scan' is at data/pd-neut-cwl_cosio-d20_scan-213f-scan.zip
Saving project to 'pd-neut-cwl_cosio-d20_scan-213f'


## Stage the template fit

The template is the 2.3 K pattern. Fit it alone first, in stages, so the scan starts from a
good model: switch the fitting mode to a single fit, and start with nothing free.

In [10]:
project.analysis.fitting_mode = 'single'
for parameter in project.free_parameters:
    parameter.free = False

In [11]:
project.display.pattern('d20')

View(html=<310,531 characters>, height=480)

## Perform the analysis

### Scale, 2θ zero and cell

In [12]:
experiment.linked_structures['cosio'].scale.free = True
experiment.instrument.calib_twotheta_offset.free = True
structure.cell.length_a.free = True
structure.cell.length_b.free = True
structure.cell.length_c.free = True

In [13]:
project.display.parameters.free()

#,datablock,category,entry,parameter,value,s.u.,min,max,units
1,cosio,cell,,length_a,10.3081,0,0,30,Å
2,cosio,cell,,length_b,6.00219,0,0,30,Å
3,cosio,cell,,length_c,4.78739,0,0,30,Å
4,d20,instrument,,calib_twotheta_offset,0.29433,0,-inf,inf,deg
5,d20,linked_structure,cosio,scale,1.81723,0,-inf,inf,


In [14]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_cosio-d20_scan-213f, 1438 pts fitted / 1506 loaded, 5 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       15.23        64.02
      1      0.043     14.22        55.83   12.8% ↓
  done · converged · 1 iters · reduced χ² 55.83 · Rwp 14.22 % · 0.05 s


View(html=<89,874 characters>, height=376)

In [15]:
project.display.pattern('d20')

View(html=<311,984 characters>, height=480)

### Background

The sixteen background points follow the level between the peaks.

In [16]:
for point in experiment.background:
    point.intensity.free = True

In [17]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_cosio-d20_scan-213f, 1438 pts fitted / 1506 loaded, 21 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       14.22        56.46
      1      0.036     13.46        50.59   10.4% ↓
  done · converged · 1 iters · reduced χ² 50.59 · Rwp 13.46 % · 0.04 s


View(html=<109,343 characters>, height=376)

In [18]:
project.display.pattern('d20')

View(html=<312,020 characters>, height=480)

### Peak widths

U, V and W only: across a temperature scan the Lorentzian term and the asymmetry stay at their
2.3 K values, as do the form factor's j2 term.

In [19]:
experiment.peak.broad_gauss_u.free = True
experiment.peak.broad_gauss_v.free = True
experiment.peak.broad_gauss_w.free = True

In [20]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_cosio-d20_scan-213f, 1438 pts fitted / 1506 loaded, 24 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       13.46        50.70
      1      0.043     13.28        49.35   2.7% ↓
  done · converged · 1 iters · reduced χ² 49.35 · Rwp 13.28 % · 0.05 s


View(html=<112,039 characters>, height=376)

In [21]:
project.display.pattern('d20')

View(html=<312,037 characters>, height=480)

### Atom positions

In [22]:
for site in structure.atom_sites:
    for coordinate in (site.fract_x, site.fract_y, site.fract_z):
        coordinate.free = not coordinate.symmetry_constrained

In [23]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_cosio-d20_scan-213f, 1438 pts fitted / 1506 loaded, 35 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       13.28        49.74
      1      0.048     13.22        49.27   0.9% ↓
  done · converged · 1 iters · reduced χ² 49.27 · Rwp 13.22 % · 0.05 s


View(html=<126,926 characters>, height=376)

In [24]:
project.display.pattern('d20')

View(html=<312,127 characters>, height=480)

### Displacements

Co2's displacement follows Co1's by the project's constraint.

In [25]:
for site in structure.atom_sites:
    site.adp_iso.free = not site.adp_iso.user_constrained

In [26]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_cosio-d20_scan-213f, 1438 pts fitted / 1506 loaded, 40 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       13.22        49.45
      1      0.046     13.00        47.83   3.3% ↓
  done · converged · 1 iters · reduced χ² 47.83 · Rwp 13.00 % · 0.05 s


View(html=<135,186 characters>, height=376)

In [27]:
project.display.pattern('d20')

View(html=<312,234 characters>, height=480)

### Magnetic moments

Last, the moments: Co1 in any direction, Co2 along b.

In [28]:
for parameter in (
    structure.atom_site_moment['Co1'].crystalaxis_x,
    structure.atom_site_moment['Co1'].crystalaxis_y,
    structure.atom_site_moment['Co1'].crystalaxis_z,
    structure.atom_site_moment['Co2'].crystalaxis_y,
):
    parameter.free = True

In [29]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_cosio-d20_scan-213f, 1438 pts fitted / 1506 loaded, 44 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       13.00        47.97
      1      0.055      4.69         6.25   87.0% ↓
      2      0.063      4.69         6.25   
      3      0.070      4.69         6.25   0.1% ↓
      4      0.077      4.69         6.24   
      5      0.083      4.69         6.24   0.1% ↓
      6      0.094      4.69         6.24   
  done · converged · 6 iters · reduced χ² 6.24 · Rwp 4.69 % · 0.10 s


View(html=<140,471 characters>, height=376)

In [30]:
project.display.pattern('d20')

View(html=<312,046 characters>, height=480)

## Fit the scan

Back to the sequential mode, with the template's free parameters. The fit can stop between
patterns: here it stops after five, all below 3 K, and the rows it wrote stay.

In [31]:
project.analysis.fitting_mode = 'sequential'
fitted = []
result = project.fit_sequential(
    on_file_complete=lambda *args: fitted.append(args),
    should_cancel=lambda: len(fitted) >= 5,
)
print(result.status.name.lower(), 'after', len(fitted), 'patterns')

cancelled after 5 patterns


## Resume

Starting again continues with the first pattern that has no row yet, from the last fitted
values. The whole scan takes a while.

In [32]:
result = project.fit_sequential(on_file_complete=lambda *args: fitted.append(args))
print(result.status.name.lower(), 'with', len(fitted), 'patterns fitted in all')

done with 213 patterns fitted in all


## The moments against temperature

One row per pattern, in the order they were fitted. The moment columns name their site and
component.

In [33]:
results = Path('pd-neut-cwl_cosio-d20_scan-213f/analysis/results.csv')
with results.open(newline='', encoding='utf-8') as file:
    rows = list(csv.DictReader(file))
moments = [name for name in rows[0] if 'crystalaxis' in name and not name.endswith('_su')]
print(len(rows), 'rows; moment columns:', ', '.join(moments))

213 rows; moment columns: cosio.atom_site_moment.Co1.crystalaxis_x, cosio.atom_site_moment.Co1.crystalaxis_x.uncertainty, cosio.atom_site_moment.Co1.crystalaxis_y, cosio.atom_site_moment.Co1.crystalaxis_y.uncertainty, cosio.atom_site_moment.Co1.crystalaxis_z, cosio.atom_site_moment.Co1.crystalaxis_z.uncertainty, cosio.atom_site_moment.Co2.crystalaxis_y, cosio.atom_site_moment.Co2.crystalaxis_y.uncertainty


## Report

The HTML report draws the measured, calculated and difference patterns with the nuclear and the
magnetic reflection positions, and lists the structure and the fit. Open it in a browser.

In [34]:
project.report.save_html('pd-neut-cwl_cosio-d20_scan-213f.html')

Saving report to 'pd-neut-cwl_cosio-d20_scan-213f.html'


## Save the project

The fitted template and the scan results are in the project folder already; save the
project once more after the fits.

In [35]:
project.save()

Saving project to 'pd-neut-cwl_cosio-d20_scan-213f'
